# 01 - Data Exploration

This notebook demonstrates how to:
1. Fetch market data using the data pipeline
2. Explore OHLCV data from TimescaleDB
3. Visualize price and volume data

In [ ]:
import sys
sys.path.insert(0, '/app')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data_pipeline.loader import DataLoader
from data_pipeline.ingestion import DataFetcher

plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.max_columns', 50)

## 1. Ingest Data (run once)

In [ ]:
fetcher = DataFetcher()

# Fetch a few tickers
tickers = ['AAPL', 'MSFT', 'GOOGL', 'SPY']
fetcher.run_full_ingestion(tickers, start_date='2020-01-01')

## 2. Load Data from TimescaleDB

In [ ]:
loader = DataLoader()

# Check available tickers
print('Available tickers:', loader.get_available_tickers())

# Load AAPL data
df = loader.load_ohlcv('AAPL', start_date='2020-01-01')
print(f'\nShape: {df.shape}')
df.head()

## 3. Visualize Price Data

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(df.index, df['close'], label='Close', linewidth=1)
axes[0].set_title('AAPL Close Price')
axes[0].set_ylabel('Price ($)')
axes[0].legend()

axes[1].bar(df.index, df['volume'], alpha=0.7, width=1)
axes[1].set_title('AAPL Volume')
axes[1].set_ylabel('Volume')

plt.tight_layout()
plt.show()

## 4. Basic Statistics

In [ ]:
# Daily returns
df['returns'] = df['close'].pct_change()

print('Return statistics:')
print(df['returns'].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['returns'].hist(bins=100, ax=axes[0], alpha=0.7)
axes[0].set_title('Return Distribution')
axes[0].axvline(0, color='red', linestyle='--')

df['returns'].cumsum().plot(ax=axes[1])
axes[1].set_title('Cumulative Returns')

plt.tight_layout()
plt.show()

## 5. Multi-Asset Comparison

In [ ]:
tickers = ['AAPL', 'MSFT', 'GOOGL', 'SPY']
dfs = loader.load_multiple(tickers, start_date='2020-01-01')

# Normalized prices (base 100)
fig, ax = plt.subplots(figsize=(14, 6))
for ticker, data in dfs.items():
    normalized = 100 * data['close'] / data['close'].iloc[0]
    ax.plot(data.index, normalized, label=ticker, linewidth=1)

ax.set_title('Normalized Price Comparison (base=100)')
ax.set_ylabel('Indexed Price')
ax.legend()
plt.tight_layout()
plt.show()